# 2:4 sparsity for the squeezed layer (features[27])

Starts from the trained dense model (92.45%), prunes the 8x8 squeezed conv to a 2:4 pattern
where in every group of 4 input channels at most 2 nonzero weights per output channel and kernel position,
fine-tunes with the mask fixed, verifies the integer data, and exports the text files for
`hardware/alpha_nm` in dense and compressed weight formats.

Requires the `nm_mask` change in `models/quant_layer.py`: the mask is applied
after quantization so pruned weights are exactly 0 in `w_int`.

In [1]:
import os, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

from models import *

DATA_DIR   = '../data'                                                 
MODEL_PATH = '../final_trained_model/VGG16_quant_8x8_92.45_model.pth'
RESULT_DIR = 'result/VGG16_quant_8x8_2to4_NM'
EXPORT_DIR = '../../hardware/alpha_nm/sys_data_nm'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

batch_size = 128
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True,
    transform=transforms.Compose([transforms.RandomCrop(32, padding=4),
                                  transforms.RandomHorizontalFlip(),
                                  transforms.ToTensor(), normalize]))
test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True,
    transform=transforms.Compose([transforms.ToTensor(), normalize]))

trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                                          num_workers=2, persistent_workers=True)
testloader  = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                                          num_workers=2, persistent_workers=True)

device: cpu


In [2]:
def evaluate(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in testloader:
            data, target = data.to(device), target.to(device)
            correct += model(data).argmax(dim=1).eq(target).sum().item()
    return 100. * correct / len(testloader.dataset)


def train_one_epoch(model, criterion, optimizer):
    model.train()
    total_loss, correct, seen = 0., 0, 0
    for data, target in trainloader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.size(0)
        correct += output.argmax(dim=1).eq(target).sum().item()
        seen += data.size(0)
    return total_loss / seen, 100. * correct / seen


def make_nm_mask(weight, N=2, M=4):
    # N:M mask along the input-channel dim: in each group of M consecutive input channels
    # for every output channel and kernel position keep the N largest |w - mean|,
    # the largest weights after the quantizer's mean-normalization
    oc, ic, kh, kw = weight.shape
    score = (weight - weight.mean()).abs().reshape(oc, ic // M, M, kh, kw)
    keep = score.topk(N, dim=2).indices
    return torch.zeros_like(score).scatter_(2, keep, 1.0).reshape(oc, ic, kh, kw)

In [3]:
## Load the trained dense model
model = VGG16_quant_8x8()
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)

print('Dense model accuracy: {:.2f}%'.format(evaluate(model)))

Dense model accuracy: 92.35%


In [ ]:
## Prune to 2:4
N, M = 2, 4
layer = model.features[27]

with torch.no_grad():
    mask = make_nm_mask(layer.weight, N, M)
layer.register_buffer('nm_mask', mask)   # applied after quantization in forward

oc, ic, kh, kw = mask.shape
print('kept fraction          :', mask.mean().item())
print('max nonzeros per group :', int(mask.reshape(oc, ic // M, M, kh, kw).sum(dim=2).max()))
print('Accuracy after 2:4 pruning (no fine-tuning): {:.2f}%'.format(evaluate(model)))

In [ ]:
## Fine-tune with the mask fixed (pruned weights get zero gradient through the mask)
lr = 0.005
weight_decay = 5e-4
epochs = 30

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
os.makedirs(RESULT_DIR, exist_ok=True)

best_prec = 0
for epoch in range(epochs):
    start = time.time()
    cur_lr = optimizer.param_groups[0]['lr']
    loss, train_acc = train_one_epoch(model, criterion, optimizer)
    prec = evaluate(model)
    scheduler.step()

    if prec > best_prec:
        best_prec = prec
        torch.save({'state_dict': model.state_dict(), 'best_prec': best_prec, 'epoch': epoch + 1},
                   os.path.join(RESULT_DIR, 'model_best.pth'))
    print('epoch {:2d}/{}  lr {:.5f}  loss {:.4f}  train {:.2f}%  test {:.2f}%  best {:.2f}%  ({:.0f}s)'.format(
          epoch + 1, epochs, cur_lr, loss, train_acc, prec, best_prec, time.time() - start))

In [ ]:
## Reload the best fine-tuned checkpoint into a fresh model
model = VGG16_quant_8x8()
model.features[27].register_buffer('nm_mask', torch.zeros(8, 8, 3, 3))  # placeholder so the saved mask loads
checkpoint = torch.load(os.path.join(RESULT_DIR, 'model_best.pth'), map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)

print('2:4 model (best epoch {}): {:.2f}%'.format(checkpoint['epoch'], evaluate(model)))

In [ ]:
## Hook features[27] (input) and features[29] (input = ReLU output of the squeezed layer)
class SaveInput:
    def __init__(self):
        self.inputs = []
    def __call__(self, module, module_in):
        self.inputs.append(module_in[0].detach().clone())

save_in, save_next = SaveInput(), SaveInput()
h1 = model.features[27].register_forward_pre_hook(save_in)
h2 = model.features[29].register_forward_pre_hook(save_next)

images, labels = next(iter(testloader))
model.eval()
with torch.no_grad():
    model(images.to(device))
h1.remove()
h2.remove()

layer = model.features[27]
x_bit, w_bit = 4, 4
with torch.no_grad():
    x = save_in.inputs[0][0]                       # image 0, [8, 4, 4]
    next_in = save_next.inputs[0][0]               # image 0, [8, 4, 4]

    x_delta = layer.act_alpha / (2 ** x_bit - 1)
    x_int = (act_quantization(x_bit)(x, layer.act_alpha) / x_delta).round()

    w_delta = layer.weight_quant.wgt_alpha / (2 ** (w_bit - 1) - 1)
    w_int = (layer.weight_q.detach() / w_delta).round()   # weight_q is stored masked by forward()

    psum_int = F.conv2d(x_int.unsqueeze(0), w_int, padding=1)[0]          # [8, 4, 4]
    psum_recovered = F.relu(psum_int * x_delta * w_delta)

    nz_per_group = (w_int != 0).reshape(8, 2, 4, 3, 3).sum(dim=2)
    print('w_int range                     :', int(w_int.min()), '..', int(w_int.max()))
    print('max nonzeros per group in w_int :', int(nz_per_group.max()))
    print('pruned positions exactly 0      :', bool((w_int[layer.nm_mask == 0] == 0).all()))
    print('mean abs diff vs next input     :', (psum_recovered - next_in).abs().mean().item())

In [ ]:
## Export text files for hardware/alpha_nm (same formats as the notebook and gen_random_nm.py)
os.makedirs(EXPORT_DIR, exist_ok=True)

X = F.pad(x_int, (1, 1, 1, 1)).reshape(8, 36).cpu().long()   # [ic, nij], padded 6x6
W = w_int.reshape(8, 8, 9).permute(2, 0, 1).cpu().long()     # [kij, oc, ic]
P = F.relu(psum_int).reshape(8, 16).cpu().long()             # [oc, onij]

b2  = lambda v: format(int(v) & 0x3, '02b')
b4  = lambda v: format(int(v) & 0xF, '04b')
b16 = lambda v: format(int(v) & 0xFFFF, '016b')

def compress(w):
    # 8 dense weights of one (kij, oc) -> 4 (idx, w) entries, array rows 0..3 (same as gen_random_nm.py)
    entries = []
    for g in range(2):
        nz = [i for i in range(4) if w[4 * g + i] != 0]
        pad = [i for i in range(4) if i not in nz][:2 - len(nz)]    # < 2 nonzeros: pad with w = 0
        for i in sorted(nz + pad):                                  # lower channel -> even row
            entries.append((i, int(w[4 * g + i])))
    return entries

with open(f'{EXPORT_DIR}/input.txt', 'w') as f:
    f.write('#time0row7[msb-lsb],time0row6[msb-lst],....,time0row0[msb-lst]#\n')
    f.write('#time1row7[msb-lsb],time1row6[msb-lst],....,time1row0[msb-lst]#\n')
    f.write('#................#\n')
    for n in range(36):
        f.write(''.join(b4(X[r, n]) for r in range(7, -1, -1)) + '\n')

with open(f'{EXPORT_DIR}/weight_dense.txt', 'w') as f:      # for the dense build (SPARSE_NM=0)
    f.write('#col0row7[msb-lsb],...,col0row0[msb-lsb]#\n')
    f.write('#col1row7[msb-lsb],...,col1row0[msb-lsb]#\n')
    f.write('#................#\n')
    for k in range(9):
        for c in range(8):
            f.write(''.join(b4(W[k, c, r]) for r in range(7, -1, -1)) + '\n')

with open(f'{EXPORT_DIR}/weight_nm.txt', 'w') as f:         # for the 2:4 build (SPARSE_NM=1)
    f.write('#2:4_compressed:col0_arrayrow3..arrayrow0,byte={00,idx,w}#\n')
    f.write('#rows2g,2g+1=kept_weights_of_channels4g..4g+3,lower_ch_on_even_row#\n')
    f.write('#................#\n')
    for k in range(9):
        for c in range(8):
            entries = compress(W[k, c])
            f.write(''.join('00' + b2(i) + b4(v) for i, v in reversed(entries)) + '\n')

with open(f'{EXPORT_DIR}/psum.txt', 'w') as f:
    f.write('#time0col7[msb-lsb],time0col6[msb-lst],....,time0col0[msb-lst]#\n')
    f.write('#time1col7[msb-lsb],time1col6[msb-lst],....,time1col0[msb-lst]#\n')
    f.write('#................#\n')
    for o in range(16):
        f.write(''.join(b16(P[c, o]) for c in range(7, -1, -1)) + '\n')

print('wrote', sorted(os.listdir(EXPORT_DIR)), 'to', EXPORT_DIR)